# Router sample record dataset builder

Load per-model Cauldron evaluation parquet data, keep every SampleRecord column as its own field, and build deterministic train/validation/test splits that save directly under the final dataset directory.

In [1]:
from pathlib import Path
from dataclasses import fields
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

from imports.config import SampleRecord
from imports.check_data_utils import (
    DEFAULT_IMAGE_ROOT,
    find_local_image_path,
    load_run_records,
)

## 1. Configure directories and knobs

In [2]:
PROJECT_ROOT = Path.cwd()
try:
    REPO_ROOT = PROJECT_ROOT.parents[2]
except IndexError:
    REPO_ROOT = PROJECT_ROOT

In [3]:
DATASET_DIR = REPO_ROOT / "dataset"
WHICH_VLM_DATASET = DATASET_DIR / "which_vlm_data"
WHICH_VLM_DATASET

PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data')

In [4]:
RUN_DIR_NAME = 'individual_datasets'
EXPERIMENT_DIR_PATH = WHICH_VLM_DATASET / RUN_DIR_NAME
TARGET_CONFIGS: Optional[List[str]] = None  # e.g., ['docvqa', 'textvqa']
RNG_SEED = 17

In [5]:
# Configurable split ratios (must sum to 1.0)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6, "Split ratios must sum to 1.0"


In [6]:
OUTPUT_BASE_DIR = (REPO_ROOT / 'dataset' / 'final_dataset').resolve()
print(OUTPUT_BASE_DIR)
DATA_DIR_PATH = EXPERIMENT_DIR_PATH
print(DATA_DIR_PATH)

/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset
/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/which_vlm_data/individual_datasets


print(f'Notebook cwd: {PROJECT_ROOT}')
print(f'Repo root guess: {REPO_ROOT}')
print(f'Output individual directory: {DATA_DIR_PATH}')
print(f'Output final directory: {OUTPUT_BASE_DIR}')
print(f'Split ratios - Train: {TRAIN_RATIO:.0%}, Val: {VAL_RATIO:.0%}, Test: {TEST_RATIO:.0%}')

if not DATA_DIR_PATH.exists():
    raise FileNotFoundError(f'Run directory not found: {DATA_DIR_PATH}')

print('\nAvailable parquet files:')
for file in sorted(DATA_DIR_PATH.glob('*.parquet')):
    print(' -', file.name)

## 2. Load SampleRecord rows from parquet

In [8]:
run_df = load_run_records(DATA_DIR_PATH, subset=TARGET_CONFIGS)
print(f'Total rows (sample, model): {len(run_df):,}')
print(f'Total columns: {len(run_df.columns)}')
run_df.head(2)

/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/which_vlm/dataset_builder/imports/check_data_utils.py:113: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


Total rows (sample, model): 534,350
Total columns: 65


,sample_id,run_id,timestamp_utc,image_path,image_bytes_hash,prompt_raw,prompt_formatted,system_prompt,source_dataset,source_config,...,semantic_f1_gt_statements,semantic_f1_matches,semantic_f1_labels,glider_score,glider_reasoning,glider_highlight,glider_raw_output,semantic_precision,semantic_recall,semantic_f1
0,ai2d_00024_ab0c600e6d10613f,exp_20251127_132944,2025-11-27T18:30:57.471038,None,ab0c600e6d10613f,Question: Which of the above things are the la...,None,None,cauldron_ai2d,ai2d,...,None,None,None,5.0,"- The model's output ""C"" is the correct answer...","[C, The Sun, largest]","<reasoning>\n- The model's output ""C"" is the ...",NaN,NaN,NaN
1,ai2d_00025_825ad24fd4599ce7,exp_20251127_132944,2025-11-27T18:31:03.026694,None,825ad24fd4599ce7,Question: What is the layer above the upper ma...,None,None,cauldron_ai2d,ai2d,...,None,None,None,0.0,- The model's output does not provide any answ...,"[does not provide, expected answer, correct ch...",<reasoning>\n- The model's output does not pr...,NaN,NaN,NaN


### Column inventory

In [9]:
column_inventory = pd.DataFrame({
    'column': run_df.columns,
    'dtype': run_df.dtypes.astype(str),
}).sort_values('column').reset_index(drop=True)
column_inventory.head(40)

,column,dtype
0,error_message,object
1,estimated_cost_usd,float64
2,glider_highlight,object
3,glider_raw_output,object
4,glider_reasoning,object
5,glider_score,float64
6,ground_truth,object
7,ground_truth_type,object
8,gt_answer_letter,object
9,image_bytes_hash,object


## 3. Ensure every `SampleRecord` attribute is present

In [10]:
sample_record_fields = [field.name for field in fields(SampleRecord)]
missing_fields = [col for col in sample_record_fields if col not in run_df.columns]
if missing_fields:
    for col in missing_fields:
        run_df[col] = pd.NA
    print('Added missing columns:', missing_fields)
else:
    print('No missing SampleRecord columns.')

ordered_columns = sample_record_fields + [col for col in run_df.columns if col not in sample_record_fields]
run_df = run_df[ordered_columns]
print(f'Columns after ordering: {len(run_df.columns)}')
run_df[sample_record_fields[:10]].head(2)

No missing SampleRecord columns.
Columns after ordering: 65


,sample_id,run_id,timestamp_utc,image_path,image_bytes_hash,prompt_raw,prompt_formatted,system_prompt,source_dataset,source_config
0,ai2d_00024_ab0c600e6d10613f,exp_20251127_132944,2025-11-27T18:30:57.471038,None,ab0c600e6d10613f,Question: Which of the above things are the la...,None,None,cauldron_ai2d,ai2d
1,ai2d_00025_825ad24fd4599ce7,exp_20251127_132944,2025-11-27T18:31:03.026694,None,825ad24fd4599ce7,Question: What is the layer above the upper ma...,None,None,cauldron_ai2d,ai2d


## 4. Helper functions for pivoting and splits

In [11]:
def create_pivot_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform long-form data into pivot table where each row is a unique sample
    and columns contain all model responses with their full SampleRecord fields.
    """
    # Get all SampleRecord fields
    sample_record_fields = [field.name for field in fields(SampleRecord)]

    # Fields that are shared across all models (same for each sample)
    shared_fields = [
        'sample_id', 'run_id', 'timestamp_utc', 'image_path', 'image_bytes_hash',
        'prompt_raw', 'prompt_formatted', 'system_prompt', 'source_dataset',
        'source_config', 'router_task', 'ground_truth', 'ground_truth_type',
        'mc_options', 'source_index', 'img_width', 'img_height', 'img_aspect_ratio',
        'img_file_size_bytes', 'txt_prompt_length_chars', 'txt_prompt_length_words',
        'txt_question_type', 'txt_has_mc_options'
    ]

    # Fields that are model-specific (need to be pivoted)
    model_specific_fields = [f for f in sample_record_fields if f not in shared_fields]

    pivot_rows = []

    # Group by sample_id
    for sample_id, group in df.groupby('sample_id'):
        # Start with shared fields from first row
        first_row = group.iloc[0]
        pivot_row = {field: first_row[field] for field in shared_fields}

        # Add metadata for tracking back to Cauldron
        pivot_row['cauldron_image_asset'] = f"{first_row['source_config']}/{first_row['image_bytes_hash']}.png"
        pivot_row['cauldron_lookup_key'] = f"{first_row['source_config']}:{first_row['source_index']}"
        pivot_row['image_cache_root'] = str(DEFAULT_IMAGE_ROOT.resolve())

        # Number of models that evaluated this sample
        pivot_row['n_models'] = len(group)

        # Add model-specific fields with model name prefix
        for idx, (_, model_row) in enumerate(group.iterrows()):
            model_name = model_row['model_name']
            # Clean model name for column naming
            clean_model_name = model_name.replace('/', '_').replace('-', '_').replace('.', '_')

            for field in model_specific_fields:
                col_name = f"{clean_model_name}__{field}"
                pivot_row[col_name] = model_row[field]

        pivot_rows.append(pivot_row)

    return pd.DataFrame(pivot_rows)


def assign_sample_splits(
    df: pd.DataFrame,
    seed: int = 0,
    train_ratio: float = 0.7,
    val_ratio: float = 0.2,
    test_ratio: float = 0.1
) -> pd.DataFrame:
    """Assign train/val/test splits to samples with configurable ratios."""
    df = df.copy()

    # Shuffle sample IDs
    sample_ids = (
        df[['sample_id']]
        .drop_duplicates()
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )

    n = len(sample_ids)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)

    # Assign splits
    sample_ids['subset_split'] = 'test'
    if train_end > 0:
        sample_ids.loc[:train_end - 1, 'subset_split'] = 'train'
    if val_end > train_end:
        sample_ids.loc[train_end:val_end - 1, 'subset_split'] = 'validation'

    # Map back to original dataframe
    split_map = dict(zip(sample_ids['sample_id'], sample_ids['subset_split']))
    df['subset_split'] = df['sample_id'].map(split_map)

    return df


def summarize_splits(df: pd.DataFrame) -> pd.DataFrame:
    """Generate summary statistics for each split."""
    summary = (
        df.groupby('subset_split')
        .agg(
            unique_samples=('sample_id', 'nunique'),
        )
        .reset_index()
        .sort_values('subset_split')
    )
    return summary

## 5. Build pivot dataset with one row per sample

In [12]:
print('Creating pivot dataset...')
print(f'Original long-form data: {len(run_df):,} rows (sample, model pairs)')
print(f'Unique samples: {run_df["sample_id"].nunique():,}')
print(f'Unique models: {run_df["model_name"].nunique()}')

# Create pivot table
pivot_df = create_pivot_dataset(run_df)
print(f'\nPivot dataset: {len(pivot_df):,} rows (unique samples)')
print(f'Total columns: {len(pivot_df.columns)}')

# Assign train/val/test splits
pivot_df = assign_sample_splits(
    pivot_df,
    seed=RNG_SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO
)

print(f'\nDataset split:')
split_summary = summarize_splits(pivot_df)
for _, row in split_summary.iterrows():
    split_name = row['subset_split']
    count = row['unique_samples']
    pct = count / len(pivot_df) * 100
    print(f'  {split_name}: {count:,} samples ({pct:.1f}%)')

pivot_df.head(2)

Creating pivot dataset...
Original long-form data: 534,350 rows (sample, model pairs)
Unique samples: 91,376
Unique models: 5

Pivot dataset: 91,376 rows (unique samples)
Total columns: 222

Dataset split:
  test: 13,707 samples (15.0%)
  train: 63,963 samples (70.0%)
  validation: 13,706 samples (15.0%)


,sample_id,run_id,timestamp_utc,image_path,image_bytes_hash,prompt_raw,prompt_formatted,system_prompt,source_dataset,source_config,...,gemma_3_27b__semantic_f1_f1,gemma_3_27b__semantic_f1_gen_statements,gemma_3_27b__semantic_f1_gt_statements,gemma_3_27b__semantic_f1_matches,gemma_3_27b__semantic_f1_labels,gemma_3_27b__glider_score,gemma_3_27b__glider_reasoning,gemma_3_27b__glider_highlight,gemma_3_27b__glider_raw_output,subset_split
0,ai2d_00000_45f9e7163ea99b4c,exp_20251127_132944,2025-11-27T18:30:57.474332,None,45f9e7163ea99b4c,Question: What do respiration and combustion g...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,4.0,- The model's output is semantically correct a...,"[B, carbon dioxide, respiration, combustion, c...",<reasoning>\n- The model's output is semantic...,train
1,ai2d_00001_0135592f21ea024d,exp_20251127_132944,2025-11-27T18:31:03.209660,None,0135592f21ea024d,"Question: From the given food web, name any tw...",None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,4.0,- The model's answer is semantically correct a...,"[Jack Rabbit, Jack Rabbit, herbivores, eats pl...",<reasoning>\n- The model's answer is semantic...,train


### Inspect one sample with all model responses

In [13]:
# Look at one sample to see the pivot structure
sample_focus = pivot_df.iloc[0]
print(f"Sample ID: {sample_focus['sample_id']}")
print(f"Source: {sample_focus['source_config']}")
print(f"Task: {sample_focus['router_task']}")
print(f"Number of models: {sample_focus['n_models']}")
print(f"Cauldron lookup: {sample_focus['cauldron_lookup_key']}")
print(f"Image asset: {sample_focus['cauldron_image_asset']}")
print(f"\nPrompt: {sample_focus['prompt_raw'][:200]}...")
print(f"Ground truth: {sample_focus['ground_truth']}")

# Show model-specific columns for this sample
model_cols = [col for col in pivot_df.columns if '__' in col]
print(f"\nTotal model-specific columns: {len(model_cols)}")
print(f"Example model columns (first 10):")
for col in model_cols[:10]:
    print(f"  - {col}")

Sample ID: ai2d_00000_45f9e7163ea99b4c
Source: ai2d
Task: diagram_reasoning
Number of models: 5
Cauldron lookup: ai2d:0.0
Image asset: ai2d/45f9e7163ea99b4c.png

Prompt: Question: What do respiration and combustion give out
Choices:
A. Oxygen
B. Carbon dioxide
C. Nitrogen
D. Heat
Answer with the letter....
Ground truth: Answer: B

Total model-specific columns: 195
Example model columns (first 10):
  - deepseek_ocr__model_name
  - deepseek_ocr__model_id
  - deepseek_ocr__response_raw
  - deepseek_ocr__response_parsed
  - deepseek_ocr__response_length_chars
  - deepseek_ocr__response_length_tokens
  - deepseek_ocr__stop_reason
  - deepseek_ocr__error_message
  - deepseek_ocr__is_refusal
  - deepseek_ocr__ok


### Detailed split summary

In [14]:
split_summary = summarize_splits(pivot_df)
print("Dataset Split Summary:")
print("=" * 50)
for _, row in split_summary.iterrows():
    split_name = row['subset_split']
    count = row['unique_samples']
    pct = count / len(pivot_df) * 100
    print(f"{split_name.capitalize():12s}: {count:6,} samples ({pct:5.1f}%)")
print("=" * 50)
print(f"{'Total':12s}: {len(pivot_df):6,} samples (100.0%)")

split_summary

Dataset Split Summary:
Test        : 13,707 samples ( 15.0%)
Train       : 63,963 samples ( 70.0%)
Validation  : 13,706 samples ( 15.0%)
Total       : 91,376 samples (100.0%)


,subset_split,unique_samples
0,test,13707
1,train,63963
2,validation,13706


## 6. Persist pivot datasets (train/val/test splits)

In [15]:
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# Save full dataset
full_parquet = OUTPUT_BASE_DIR / 'router_pivot_dataset_full.parquet'
pivot_df.to_parquet(full_parquet, index=False)
print(f'Saved full dataset: {full_parquet}')
print(f'  Rows: {len(pivot_df):,}')
print(f'  Columns: {len(pivot_df.columns)}')
print(f'  File size: {full_parquet.stat().st_size / (1024**2):.2f} MB')

# Save split datasets
split_paths: Dict[str, Path] = {}
print('\nSaving split datasets:')
for split_name in ['train', 'validation', 'test']:
    split_df = pivot_df[pivot_df['subset_split'] == split_name].reset_index(drop=True)
    split_path = OUTPUT_BASE_DIR / f'router_pivot_dataset_{split_name}.parquet'
    split_df.to_parquet(split_path, index=False)
    split_paths[split_name] = split_path
    
    print(f'  {split_name.capitalize():12s}: {split_path.name}')
    print(f'    Rows: {len(split_df):,}')
    print(f'    File size: {split_path.stat().st_size / (1024**2):.2f} MB')

print('\n' + '=' * 60)
print('Dataset creation complete!')
print('=' * 60)

Saved full dataset: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_pivot_dataset_full.parquet
  Rows: 91,376
  Columns: 223
  File size: 302.24 MB

Saving split datasets:
  Train       : router_pivot_dataset_train.parquet
    Rows: 63,963
    File size: 212.43 MB
  Validation  : router_pivot_dataset_validation.parquet
    Rows: 13,706
    File size: 46.84 MB
  Test        : router_pivot_dataset_test.parquet
    Rows: 13,707
    File size: 46.40 MB

Dataset creation complete!


### Reload and validate splits

In [16]:
# Reload and validate train split
train_df = pd.read_parquet(split_paths['train'])
print(f'Train dataset validation:')
print(f'  Rows: {len(train_df):,}')
print(f'  Columns: {len(train_df.columns)}')
print(f'  Unique samples: {train_df["sample_id"].nunique():,}')
print(f'\nFirst 2 rows of train dataset:')
display(train_df.head(2))

# Show column breakdown
model_specific_cols = [col for col in train_df.columns if '__' in col]
shared_cols = [col for col in train_df.columns if '__' not in col]
print(f'\nColumn breakdown:')
print(f'  Shared columns: {len(shared_cols)}')
print(f'  Model-specific columns: {len(model_specific_cols)}')
print(f'  Total: {len(train_df.columns)}')

Train dataset validation:
  Rows: 63,963
  Columns: 223
  Unique samples: 63,963

First 2 rows of train dataset:


,sample_id,run_id,timestamp_utc,image_path,image_bytes_hash,prompt_raw,prompt_formatted,system_prompt,source_dataset,source_config,...,gemma_3_27b__semantic_f1_f1,gemma_3_27b__semantic_f1_gen_statements,gemma_3_27b__semantic_f1_gt_statements,gemma_3_27b__semantic_f1_matches,gemma_3_27b__semantic_f1_labels,gemma_3_27b__glider_score,gemma_3_27b__glider_reasoning,gemma_3_27b__glider_highlight,gemma_3_27b__glider_raw_output,subset_split
0,ai2d_00000_45f9e7163ea99b4c,exp_20251127_132944,2025-11-27T18:30:57.474332,None,45f9e7163ea99b4c,Question: What do respiration and combustion g...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,4.0,- The model's output is semantically correct a...,"[B, carbon dioxide, respiration, combustion, c...",<reasoning>\n- The model's output is semantic...,train
1,ai2d_00001_0135592f21ea024d,exp_20251127_132944,2025-11-27T18:31:03.209660,None,0135592f21ea024d,"Question: From the given food web, name any tw...",None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,4.0,- The model's answer is semantically correct a...,"[Jack Rabbit, Jack Rabbit, herbivores, eats pl...",<reasoning>\n- The model's answer is semantic...,train



Column breakdown:
  Shared columns: 28
  Model-specific columns: 195
  Total: 223


In [17]:
train_df.columns.tolist()

['sample_id',
 'run_id',
 'timestamp_utc',
 'image_path',
 'image_bytes_hash',
 'prompt_raw',
 'prompt_formatted',
 'system_prompt',
 'source_dataset',
 'source_config',
 'router_task',
 'ground_truth',
 'ground_truth_type',
 'mc_options',
 'source_index',
 'img_width',
 'img_height',
 'img_aspect_ratio',
 'img_file_size_bytes',
 'txt_prompt_length_chars',
 'txt_prompt_length_words',
 'txt_question_type',
 'txt_has_mc_options',
 'cauldron_image_asset',
 'cauldron_lookup_key',
 'image_cache_root',
 'n_models',
 'deepseek_ocr__model_name',
 'deepseek_ocr__model_id',
 'deepseek_ocr__response_raw',
 'deepseek_ocr__response_parsed',
 'deepseek_ocr__response_length_chars',
 'deepseek_ocr__response_length_tokens',
 'deepseek_ocr__stop_reason',
 'deepseek_ocr__error_message',
 'deepseek_ocr__is_refusal',
 'deepseek_ocr__ok',
 'deepseek_ocr__score_exact_match',
 'deepseek_ocr__score_exact_match_normalized',
 'deepseek_ocr__score_contains_gt',
 'deepseek_ocr__score_gt_in_response',
 'deepseek_oc

In [18]:
print("Average Latency DeepSeek OCR:", train_df['deepseek_ocr__latency_ms'].mean())
print("Average Latency Qwen2.5 VL 3B:", train_df['qwen2_5_vl_3b__latency_ms'].mean())
print("Average Latency Qwen3 VL 8B Thinking:", train_df['qwen3_vl_8b_thinking__latency_ms'].mean())
print("Average Latency Gemma 3 27B:", train_df['gemma_3_27b__latency_ms'].mean())
print("Average Latency Qwen2.5 7B:", train_df['qwen2_5_vl_7b__latency_ms'].mean())
      
    #   train_df[''].mean(), train_df['qwen3_vl_8b_thinking__latency_ms'].mean(), train_df['gemma_3_27b__latency_ms'].mean()

Average Latency DeepSeek OCR: 879.1211613153123
Average Latency Qwen2.5 VL 3B: 993.0918620238542
Average Latency Qwen3 VL 8B Thinking: 3234.4181765449625
Average Latency Gemma 3 27B: 2714.075618771062
Average Latency Qwen2.5 7B: 1454.957180394596


In [19]:
# from imports.check_data_utils import fetch_cauldron_image
# image, sample_dict = fetch_cauldron_image(
#     "ai2d",
#     24,
#     image_hash="ab0c600e6d10613f",
# )
# display(image)
